# Cursive Transformer — Train & Generate (Colab)

Modernized model: RoPE · RMSNorm · SwiGLU · fused FlashAttention · classifier-free guidance · nucleus + structured sampling.

**Before you start:** `Runtime → Change runtime type → GPU` (an **A100** is ideal).

Steps: 1) check GPU → 2) set config → 3) clone + install → 4) train → 5) generate.

> The architecture changed vs. the original repo, so you **train from scratch** — old checkpoints won't load.
> Also: the `N_LAYER` / `N_EMBD` / `N_CTX_HEAD` you train with **must** match at inference, or the checkpoint won't load.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 2. Configuration

Fill these in. Get your W&B key at https://wandb.ai/authorize .

`REPO_URL` / `BRANCH` should point at **your** fork containing the modernized code. If you haven't pushed it, use the *Manual upload* cell further down instead.

The **architecture** values are defined here once and reused by both the train and generate cells, so they can't drift out of sync.

In [ ]:
REPO_URL      = 'https://github.com/ariedotcodotnz/cursivetransformer.git'  #@param
BRANCH        = 'modernize'                #@param

WANDB_ENTITY  = 'your-wandb-username'       #@param
WANDB_PROJECT = 'cursive_modern'           #@param
WANDB_API_KEY = ''                         #@param  (paste your key)
RUN_NAME      = 'modern_v1'                #@param

DATASET       = 'bigbank_3500'             #@param  (must be a .json.zip in data/)
NUM_WORDS     = 4                           #@param
MAX_SEQ_LEN   = 1050                        #@param

# --- model architecture: these MUST be identical for training and inference ---
N_LAYER       = 5                           #@param
N_EMBD        = 64                          #@param
N_CTX_HEAD    = 4                           #@param

import os
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
print('config set.')

## 3. Clone the repo + install dependencies

In [ ]:
import os
if not os.path.isdir('cursivetransformer'):
    !git clone -b {BRANCH} {REPO_URL}
%cd cursivetransformer
!pip -q install -r requirements.txt
print('\nDatasets available in data/:')
!ls data/*.json.zip

### (Optional) Manual upload — only if you didn't push to GitHub
Clone the **original** repo, then run this cell and upload your local `model.py`, `data.py`, `train.py`, `sample.py` to overwrite the four files.

In [ ]:
# from google.colab import files
# uploaded = files.upload()           # select model.py, data.py, train.py, sample.py
# for name, content in uploaded.items():
#     with open(name, 'wb') as f: f.write(content)
# print('overwrote:', list(uploaded))

## 4. Train

Logs loss + sample images to W&B every `log_every` steps and saves the best checkpoint as a W&B artifact (and to `best_checkpoint.pt`). On an A100 this is ~1.5–2 hours for 125k steps (~41 ms/step).

New knobs: `--cond_drop_prob 0.1` enables classifier-free guidance; `--subnetwork_mode full` trains the full model every step (best single-model quality).

In [ ]:
!python train.py \
  --wandb_entity {WANDB_ENTITY} \
  --wandb_project {WANDB_PROJECT} \
  --wandb_api_key {WANDB_API_KEY} \
  --wandb_run_name {RUN_NAME} \
  --dataset_name {DATASET} \
  --num_words {NUM_WORDS} \
  --max_seq_length {MAX_SEQ_LEN} \
  --batch_size 32 \
  --n_layer {N_LAYER} \
  --n_embd {N_EMBD} \
  --n_ctx_head {N_CTX_HEAD} \
  --learning_rate 1e-2 \
  --step_lr_every 20000 --lr_decay 0.5 \
  --max_steps 125000 \
  --train_size 497000 --test_size 3000 \
  --log_every 2500 \
  --downsample_mean 0.65 \
  --cond_drop_prob 0.1 \
  --subnetwork_mode full \
  --seed 1337

### Resume a dropped run
If Colab killed your A100 mid-run, set `RESUME_RUN_ID` to the W&B run id and re-run. It restores model + optimizer + scheduler + step. (Architecture flags are reused from the config cell, so they automatically match.)

In [ ]:
RESUME_RUN_ID = ''  #@param
if RESUME_RUN_ID:
    !python train.py \
      --wandb_entity {WANDB_ENTITY} --wandb_project {WANDB_PROJECT} --wandb_api_key {WANDB_API_KEY} \
      --dataset_name {DATASET} --num_words {NUM_WORDS} --max_seq_length {MAX_SEQ_LEN} \
      --batch_size 32 --n_layer {N_LAYER} --n_embd {N_EMBD} --n_ctx_head {N_CTX_HEAD} --learning_rate 1e-2 \
      --step_lr_every 20000 --lr_decay 0.5 --max_steps 125000 \
      --train_size 497000 --test_size 3000 --log_every 2500 \
      --downsample_mean 0.65 --cond_drop_prob 0.1 --subnetwork_mode full --seed 1337 \
      --load_from_run_id {RESUME_RUN_ID}
else:
    print('Set RESUME_RUN_ID above to resume.')

## 5. Generate handwriting

Set `RUN_ID` to a trained run (it downloads that run's best checkpoint). The architecture is pulled from the config cell so it matches the checkpoint. First load the model + tokenizer once, then generate any text you like.

In [ ]:
RUN_ID = ''  #@param  (a finished/in-progress W&B run id with a saved checkpoint)

import torch
from model import get_all_args, get_checkpoint
from data import create_datasets
from sample import GenerationParams, generate_paragraph, plot_paragraph

args = get_all_args(use_argparse=False)
args.device          = 'cuda' if torch.cuda.is_available() else 'cpu'
args.dataset_name    = DATASET
args.num_words       = NUM_WORDS
args.max_seq_length  = MAX_SEQ_LEN
args.n_layer         = N_LAYER      # must match the trained checkpoint
args.n_embd          = N_EMBD
args.n_ctx_head      = N_CTX_HEAD
args.wandb_entity    = WANDB_ENTITY
args.wandb_project   = WANDB_PROJECT
args.wandb_api_key   = WANDB_API_KEY
args.load_from_run_id = RUN_ID

train_ds, test_ds = create_datasets(args)
args.vocab_size         = train_ds.get_vocab_size()
args.block_size         = train_ds.get_stroke_seq_length()
args.context_block_size = train_ds.get_text_seq_length()
args.context_vocab_size = train_ds.get_char_vocab_size()

model, *_ = get_checkpoint(args, sample_only=True)
model.eval()
print('model loaded.')

In [ ]:
#@title Generate from custom text
TEXT           = 'the quick brown fox jumps over the lazy dog'  #@param
TEMPERATURE    = 0.8   #@param
TOP_P          = 0.95  #@param
GUIDANCE_SCALE = 2.0   #@param  (>1 sharpens spelling/text adherence)
STRUCTURED     = True  #@param  (forbid tokens that break the stroke pairing)

params = GenerationParams(
    do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
    guidance_scale=GUIDANCE_SCALE, structured=STRUCTURED,
)

offsets = generate_paragraph(model, test_ds, TEXT, params)
fig, ax = plot_paragraph(offsets, TEXT, params=params, include_title=True)
fig

### Fix a misspelled word
Pass the previous `offsets` back in and list the word indices to regenerate (set `show_indices=True` when plotting to read the index numbers).

In [ ]:
REGENERATE_IXS = [2, 4]  #@param
offsets = generate_paragraph(model, test_ds, TEXT, params,
                             word_list_offsets=offsets, regenerate_ixs=REGENERATE_IXS)
fig, ax = plot_paragraph(offsets, TEXT, params=params, show_indices=True, include_title=True)
fig